# `mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`

In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

# `Configuration (ENV)`

In [ ]:
#@title Load configuration (`.env` + Colab secrets)
#@markdown Precedence: process env > `.env` file > Colab secrets > defaults. Keys are documented in `.env.example`.<br>
#@markdown `.env` is read from `$ENV_FILE`, `/content/.env`, or `/content/drive/MyDrive/.env` (first match).
import os
from pathlib import Path
from google.colab import userdata

if "_BASE_ENV_KEYS" not in globals():
  _BASE_ENV_KEYS = set(os.environ)  # never let .env override the real process env


def load_dotenv(path):
  for raw in Path(path).read_text(encoding="utf-8").splitlines():
    line = raw.strip()
    if not line or line.startswith("#") or "=" not in line:
      continue
    key, _, value = line.removeprefix("export ").partition("=")
    key, value = key.strip(), value.strip()
    if len(value) >= 2 and value[0] == value[-1] and value[0] in "'\"":
      value = value[1:-1]
    if key not in _BASE_ENV_KEYS:
      os.environ[key] = value  # re-running picks up edits to .env


def env(key, default=None, required=False):
  value = os.environ.get(key)
  if not value:
    try:
      value = userdata.get(key)
    except (userdata.SecretNotFoundError, userdata.TimeoutException):
      value = None  # NotebookAccessError propagates: the secret exists but access was denied
  if not value:
    if required:
      raise KeyError(f"Missing config '{key}': set it in .env or as a Colab secret (see .env.example).")
    return default
  return value


def env_bool(key, default):
  value = env(key)
  return default if value is None else value.strip().lower() in ("1", "true", "yes", "on")


ENV_CANDIDATES = [os.environ.get("ENV_FILE", ""), "/content/.env", "/content/drive/MyDrive/.env"]
ENV_PATH = next((p for p in ENV_CANDIDATES if p and os.path.isfile(p)), None)
if ENV_PATH:
  load_dotenv(ENV_PATH)
  print("Loaded", ENV_PATH)
else:
  print("No .env file found; using Colab secrets and defaults.")

SUDO = "" if os.geteuid() == 0 else "sudo"
for _bin in (os.path.expanduser("~/.bun/bin"), os.path.expanduser("~/.local/bin")):  # home-installed tools
  if _bin not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = _bin + os.pathsep + os.environ["PATH"]
DRIVE_PATH = env("DRIVE_PATH", "").rstrip("/")
WORKSPACE = f"{DRIVE_PATH}/ComfyUI" if DRIVE_PATH else "/content/ComfyUI"
COMFY_PORT = int(env("COMFY_PORT", "8188"))
print(f"WORKSPACE={WORKSPACE}  COMFY_PORT={COMFY_PORT}")

# `Persistent Home (Drive)`

In [ ]:
#@title Persistent home: restore from Drive + auto-backup (restic)
#@markdown On a fresh VM restores the latest snapshot of `PERSIST_PATHS` (default `$HOME` = `/root`), then backs up every `PERSIST_INTERVAL_MIN` minutes.<br>
#@markdown Needs Drive mounted and `RESTIC_PASSWORD` (losing it makes the backups unreadable). Log: `/content/persist.log`.
import bz2
import json
import shlex
import shutil
import subprocess
import threading
import time
import urllib.request


def _env_list(key, default):
  return [os.path.expanduser(p.strip()) for p in env(key, default).split(",") if p.strip()]


RESTIC_VERSION = env("RESTIC_VERSION", "0.19.1")
PERSIST_DIR = env("PERSIST_DIR", "/content/drive/MyDrive/colab-persist").rstrip("/")
PERSIST_PATHS = _env_list("PERSIST_PATHS", "~")
PERSIST_EXCLUDES = _env_list("PERSIST_EXCLUDES",
                             "~/.cache,~/.config/Google,~/.ipython,~/.local/share/jupyter,~/.npm/_cacache,~/.bun/install/cache")
PERSIST_INTERVAL_MIN = float(env("PERSIST_INTERVAL_MIN", "10"))
PERSIST_KEEP = int(env("PERSIST_KEEP", "10"))
PERSIST_APT_PACKAGES = env("PERSIST_APT_PACKAGES", "")
PERSIST_HOST = "colab"  # fixed: Colab hostnames change per VM and restic picks parent snapshots by host
PERSIST_LOG = "/content/persist.log"
RESTORE_TARGET = "/"
RESTORED_MARKER = "/tmp/.persist-restored"  # /tmp lives as long as the VM: restore once per fresh runtime

if PERSIST_DIR.startswith("/content/drive/") and not os.path.isdir("/content/drive/MyDrive"):
  raise RuntimeError("Google Drive is not mounted: run the Drive cell first.")

if not shutil.which("restic"):
  url = f"https://github.com/restic/restic/releases/download/v{RESTIC_VERSION}/restic_{RESTIC_VERSION}_linux_amd64.bz2"
  with urllib.request.urlopen(url) as resp:
    Path("/usr/local/bin/restic").write_bytes(bz2.decompress(resp.read()))
  os.chmod("/usr/local/bin/restic", 0o755)

_RESTIC_ENV = {**os.environ, "RESTIC_REPOSITORY": f"{PERSIST_DIR}/restic",
               "RESTIC_PASSWORD": env("RESTIC_PASSWORD", required=True)}


def _restic(*args):
  cmd = ["restic", *args]
  r = subprocess.run(cmd, env=_RESTIC_ENV, text=True, capture_output=True)
  with open(PERSIST_LOG, "a") as log:
    log.write(f"\n[{time.strftime('%F %T')}] $ {shlex.join(cmd)} -> exit {r.returncode}\n{r.stdout}{r.stderr}")
  return r


def persist_restore():
  r = _restic("snapshots", "--host", PERSIST_HOST, "--json")
  if r.returncode:
    raise RuntimeError(f"Cannot list snapshots: {r.stderr.strip()}")
  snapshots = json.loads(r.stdout or "[]")
  if not snapshots:
    print("No snapshot yet: nothing to restore.")
    return
  for snap in reversed(snapshots):  # newest first; older ones are fallbacks for an incomplete Drive upload
    print(f"Restoring snapshot {snap['short_id']} ({snap['time'][:19]}): {', '.join(snap['paths'])}")
    if _restic("restore", snap["id"], "--target", RESTORE_TARGET).returncode == 0:
      return
    print(f"  restore failed (see {PERSIST_LOG}); trying an older snapshot")
  raise RuntimeError(f"No snapshot could be restored; see {PERSIST_LOG}")


_PERSIST_LOCK = threading.Lock()


def persist_save():
  """Back up PERSIST_PATHS; returns True on success (exit 3 = some files unreadable, snapshot still written)."""
  args = ["backup", "--host", PERSIST_HOST, "--one-file-system", "--exclude-caches", "--skip-if-unchanged"]
  for pattern in PERSIST_EXCLUDES:
    args += ["--exclude", pattern]
  with _PERSIST_LOCK:
    return _restic(*args, *[p for p in PERSIST_PATHS if os.path.exists(p)]).returncode in (0, 3)


def _persist_loop(stop):
  pruned = False
  while not stop.wait(PERSIST_INTERVAL_MIN * 60):
    if not persist_save():
      continue
    args = ["forget", "--host", PERSIST_HOST, "--keep-last", str(PERSIST_KEEP)]
    with _PERSIST_LOCK:  # forget is cheap; --prune rewrites packs on Drive, so only once per session
      pruned = _restic(*args, *([] if pruned else ["--prune"])).returncode == 0 or pruned


os.makedirs(PERSIST_DIR, exist_ok=True)
if not os.path.exists(f"{PERSIST_DIR}/restic/config"):
  r = _restic("init")
  if r.returncode:
    raise RuntimeError(f"restic init failed: {r.stderr.strip()}")
  print("Initialized restic repository at", f"{PERSIST_DIR}/restic")

if os.path.exists(RESTORED_MARKER):
  print("Already restored on this VM; skipping restore.")
else:
  persist_restore()
  Path(RESTORED_MARKER).touch()

if PERSIST_APT_PACKAGES:
  !{SUDO} apt-get install -y -qq {PERSIST_APT_PACKAGES} || ({SUDO} apt-get update -qq && {SUDO} apt-get install -y -qq {PERSIST_APT_PACKAGES})

if globals().get("_PERSIST_STOP"):
  _PERSIST_STOP.set()  # re-run: replace the previous backup thread
_PERSIST_STOP = threading.Event()
threading.Thread(target=_persist_loop, args=(_PERSIST_STOP,), daemon=True).start()
print(f"Auto-backup every {PERSIST_INTERVAL_MIN:g} min: {', '.join(PERSIST_PATHS)} -> {PERSIST_DIR}/restic")

In [ ]:
#@title Save home to Drive now (run before closing the session)
print("Snapshot saved." if persist_save() else f"Backup failed; see {PERSIST_LOG}")
!tail -n 6 {PERSIST_LOG}

# `Setup And Update ComfyUI`

In [ ]:
#@title Setup / update ComfyUI
#@markdown Uses `DRIVE_PATH` and `COMFY_UPDATE` from the configuration cell.
if not os.path.isdir(f"{WORKSPACE}/.git"):
  print("-= Cloning ComfyUI =-")
  !git clone https://github.com/comfyanonymous/ComfyUI "{WORKSPACE}"
elif env_bool("COMFY_UPDATE", True):
  print("-= Updating ComfyUI =-")
  !git -C "{WORKSPACE}" pull --ff-only

%cd {WORKSPACE}
print("-= Installing dependencies =-")
# Colab ships a CUDA build of torch; requirements.txt keeps it as-is.
!pip install -q -r requirements.txt

# Optional custom nodes (uncomment to install):
# !git clone https://github.com/ltdrdata/ComfyUI-Manager "{WORKSPACE}/custom_nodes/ComfyUI-Manager"
# !git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack "{WORKSPACE}/custom_nodes/ComfyUI-Impact-Pack"

# `Models Download`

## Download models (manifest of models in use — HF direct)

In [ ]:
#@title Download models — Hugging Face direct (verified)
#@markdown Groups come from `MODEL_GROUPS` (comma-separated). ~76 GB if all are enabled: set `DRIVE_PATH` so downloads survive runtime resets.
# Groups:
#   qwen-image    ~28.7G  Qwen-Image-2512 fp8 + qwen2.5-vl 7B encoder + VAE + Lightning LoRA
#   trellis       ~6.8G   TRELLIS.2 int8 + shape/texture VAE
#   pixal3d       ~6.3G   Pixal3D int8 + dino_v3 clip_vision
#   3d-aux        ~9.7G   qwen3vl 8B encoder + birefnet + moge2 (aux for TRELLIS.2/Pixal3D)
#   acestep       ~9.4G   ACE-Step 1.5 turbo + qwen 0.6B/1.7B encoders + VAE
#   minimax-music ~4.8G   MiniMax Music 3 DiT fp16 + dav VAE
#   stable-audio  ~9.7G   Stable Audio 3 medium + t5gemma encoder
#   qwen3.5       ~4.2G   qwen3.5 2B bf16 text encoder
%cd {WORKSPACE}
!command -v aria2c >/dev/null || apt-get -y install -qq aria2

DEFAULT_MODEL_GROUPS = "qwen-image,trellis,pixal3d,3d-aux,acestep,minimax-music,stable-audio"
ARIA2_JOBS = int(env("ARIA2_JOBS", "4"))
# --file-allocation=none: preallocation writes the whole file twice on Google Drive FUSE.
ARIA2_OPTS = "--console-log-level=error -c -x 16 -s 16 -k 1M --file-allocation=none --auto-file-renaming=false"

# (group, models/<subdir>, filename, url, ~GB)
MODELS = [
  ("qwen-image", "diffusion_models", "qwen_image_2512_fp8_e4m3fn.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors", 19.0),
  ("qwen-image", "text_encoders", "qwen_2.5_vl_7b_fp8_scaled.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors", 8.7),
  ("qwen-image", "vae", "qwen_image_vae.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors", 0.2),
  ("qwen-image", "loras", "Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors",
   "https://huggingface.co/lightx2v/Qwen-Image-2512-Lightning/resolve/main/Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors", 0.8),

  ("trellis", "diffusion_models", "trellis_2_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/diffusion_models/trellis_2_int8_convrot.safetensors", 4.9),
  ("trellis", "vae", "trellis_2_shape_vae_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/vae/trellis_2_shape_vae_bf16.safetensors", 1.0),
  ("trellis", "vae", "trellis_2_texture_vae_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/vae/trellis_2_texture_vae_bf16.safetensors", 0.9),

  ("pixal3d", "diffusion_models", "pixal3d_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/Pixal3D/resolve/main/diffusion_models/pixal3d_int8_convrot.safetensors", 5.2),
  ("pixal3d", "clip_vision", "dino_v3_L_naf_fp32.safetensors",
   "https://huggingface.co/Comfy-Org/Pixal3D/resolve/main/clip_vision/dino_v3_L_naf_fp32.safetensors", 1.1),

  ("3d-aux", "text_encoders", "qwen3vl_8b_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors", 8.7),
  ("3d-aux", "background_removal", "birefnet.safetensors",
   "https://huggingface.co/Comfy-Org/BiRefNet/resolve/main/background_removal/birefnet.safetensors", 0.4),
  ("3d-aux", "geometry_estimation", "moge_2_vitl_normal_fp16.safetensors",
   "https://huggingface.co/Comfy-Org/MoGe/resolve/main/geometry_estimation/moge_2_vitl_normal_fp16.safetensors", 0.6),

  ("acestep", "diffusion_models", "acestep_v1.5_turbo.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/diffusion_models/acestep_v1.5_turbo.safetensors", 4.5),
  ("acestep", "text_encoders", "qwen_0.6b_ace15.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/text_encoders/qwen_0.6b_ace15.safetensors", 1.1),
  ("acestep", "text_encoders", "qwen_1.7b_ace15.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/text_encoders/qwen_1.7b_ace15.safetensors", 3.5),
  ("acestep", "vae", "ace_1.5_vae.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/vae/ace_1.5_vae.safetensors", 0.3),

  ("minimax-music", "diffusion_models", "minimax_music3_dit_fp16.safetensors",
   "https://huggingface.co/Comfy-Org/MiniMax-Music-3/resolve/main/diffusion_models/minimax_music3_dit_fp16.safetensors", 4.6),
  ("minimax-music", "vae", "minimax_music3_dav.safetensors",
   "https://huggingface.co/Comfy-Org/MiniMax-Music-3/resolve/main/vae/minimax_music3_dav.safetensors", 0.2),

  ("stable-audio", "checkpoints", "stable_audio_3_medium.safetensors",
   "https://huggingface.co/Comfy-Org/stable-audio-3/resolve/main/checkpoints/stable_audio_3_medium.safetensors", 8.6),
  ("stable-audio", "text_encoders", "t5gemma_b_b_ul2.safetensors",
   "https://huggingface.co/Comfy-Org/stable-audio-3/resolve/main/text_encoders/t5gemma_b_b_ul2.safetensors", 1.1),

  ("qwen3.5", "text_encoders", "qwen3.5_2b_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen3.5/resolve/main/text_encoders/qwen3.5_2b_bf16.safetensors", 4.2),
]


def _is_complete(path):
  return os.path.exists(path) and not os.path.exists(path + ".aria2")  # .aria2 = unfinished download


def _with_auth(url):
  if "civitai.com" in url:
    token = env("CIVITAI_API_TOKEN")
    if token:
      url += ("&" if "?" in url else "?") + f"token={token}"
  return url


def downloadModel(url, destdir, filename=None):
  """Ad-hoc download into models/<destdir>. Civitai names come from Content-Disposition when filename is None."""
  dest = f"{WORKSPACE}/models/{destdir}"
  os.makedirs(dest, exist_ok=True)
  if filename is None and "civitai.com" not in url:
    filename = url.split("?")[0].rsplit("/", 1)[-1]
  if filename and _is_complete(f"{dest}/{filename}"):
    print("skip (exists):", filename)
    return
  out = f'-o "{filename}"' if filename else ""
  !aria2c {ARIA2_OPTS} "{_with_auth(url)}" -d "{dest}" {out}


groups = {g.strip() for g in env("MODEL_GROUPS", DEFAULT_MODEL_GROUPS).split(",") if g.strip()}
unknown = groups - {m[0] for m in MODELS}
if unknown:
  print("WARNING: unknown MODEL_GROUPS entries ignored:", ", ".join(sorted(unknown)))

selected = [m for m in MODELS if m[0] in groups]
pending = [m for m in selected if not _is_complete(f"{WORKSPACE}/models/{m[1]}/{m[2]}")]
print(f"Selected {len(selected)} files (~{sum(m[4] for m in selected):.1f} GB), "
      f"{len(pending)} to download (~{sum(m[4] for m in pending):.1f} GB).\n")

if pending:
  entries = []
  for group, destdir, filename, url, gb in pending:
    print(f"[{group}] {filename} (~{gb}G) -> models/{destdir}")
    os.makedirs(f"{WORKSPACE}/models/{destdir}", exist_ok=True)
    entries += [_with_auth(url), f"  dir={WORKSPACE}/models/{destdir}", f"  out={filename}"]
  Path("/tmp/aria2-models.txt").write_text("\n".join(entries) + "\n")
  !aria2c {ARIA2_OPTS} -j {ARIA2_JOBS} -i /tmp/aria2-models.txt

# Ad-hoc downloads (Hugging Face / Civitai):
# downloadModel('https://civitai.com/api/download/models/2071650', 'checkpoints')  # needs CIVITAI_API_TOKEN
# downloadModel('https://huggingface.co/<repo>/resolve/main/model.safetensors', 'checkpoints')

## List models

In [ ]:
!find "{WORKSPACE}/models" -type f -not -name 'put_*' -exec du -h {{}} + | sort -rh | head -40

# `Start ComfyUI`

In [ ]:
#@title Start ComfyUI
#@markdown Served through Colab's built-in port proxy: click the link printed once ComfyUI is ready.
import socket
import threading
import time

from google.colab import output


def _open_when_ready(port, timeout=600):
  deadline = time.time() + timeout
  while time.time() < deadline:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
      sock.settimeout(1)
      if sock.connect_ex(("127.0.0.1", port)) == 0:
        output.serve_kernel_port_as_window(port, anchor_text=f"Open ComfyUI (port {port})")
        return
    time.sleep(0.5)
  print("\nComfyUI was not ready after 10 minutes.\n")


threading.Thread(target=_open_when_ready, args=(COMFY_PORT,), daemon=True).start()

%cd {WORKSPACE}
!python main.py --dont-print-server --port {COMFY_PORT}

# `Developer Tools (optional)`

In [ ]:
%%bash
# Install GitHub CLI (gh) and omp (https://omp.sh) into ~/.local/bin, so the persistent home keeps them.
set -e
export PATH="$HOME/.local/bin:$PATH"
mkdir -p "$HOME/.local/bin"
if ! command -v gh >/dev/null; then
  url=$(curl -fsSLI -o /dev/null -w '%{url_effective}' https://github.com/cli/cli/releases/latest)
  ver=${url##*/v}
  curl -fsSL "https://github.com/cli/cli/releases/download/v${ver}/gh_${ver}_linux_amd64.tar.gz" | tar -xz -C /tmp
  install -m 755 "/tmp/gh_${ver}_linux_amd64/bin/gh" "$HOME/.local/bin/gh"
fi
gh --version
command -v omp >/dev/null || curl -fsSL https://omp.sh/install | sh